# 05 Main Chatbot Loop
This is the heart of the AI Therapist. It coordinates input, analysis, memory, and AI generation.

### Typical User Flow:
1. **Input**: "I had a bad day at work."
2. **Analysis**: Detects *Negative Sentiment* and *Work Concern*.
3. **Memory**: Updates the `memory.json` to remember this user has work issues.
4. **Synthesis**: Builds a prompt: *"Alex is feeling bad about work. Reflect this back gently."*
5. **Generation**: AI responds: *"I'm sorry to hear work was tough, Alex. Want to share more?"*

In [ ]:
%pip install textblob groq python-dotenv

In [1]:
# 1. Import all modules using Jupyter's %run magic command
print("Initializing AI Therapist system...")
%run "01_Memory_Module_v2.ipynb"
%run "02_Analysis_Module.ipynb"
%run "03_Prompt_Builder.ipynb"
%run "04_LLM_Interaction.ipynb"
print("Initialization complete. You can now start the therapy session.")

Initializing AI Therapist system...
Initialization complete. You can now start the therapy session.


In [2]:
def start_therapy_session():
    """
    Main loop for the therapy session.
    Handles turn-taking, memory persistence, and orchestration.
    """
    # Step A: Setup persistence
    load_long_term_memory()
    long_term_data["session_count"] = long_term_data.get("session_count", 0) + 1
    
    print(f"\n--- [AI Therapist Session #{long_term_data['session_count']}] ---")
    print(f"Hello, {long_term_data.get('user_name', 'User')}. I am here to listen. (Type 'quit' to end)\n")
    
    # Initial AI greeting to set the tone
    welcome_msg = "I'm glad you're here. How has your day been truly feeling?"
    print(f"AI: {welcome_msg}")
    add_to_history("assistant", welcome_msg)

    while True:
        # 1. Input Phase
        user_input = input("You: ").strip()
        
        if user_input.lower() in ["quit", "exit", "bye"]:
            save_long_term_memory() # Ensure everything is written to disk before closing
            print("AI: You've made good progress today. Take care of yourself.")
            break
        
        if not user_input:
            continue # Ignore empty messages

        # 2. Analysis Phase (NLP)
        # Processes the raw text to find hidden emotional cues and psychological patterns
        analysis = run_full_analysis(user_input)
        
        # 3. Long-term Memory Synchronization
        # We only save 'Insights' (emotions/concerns) to keep the JSON file focused and meaningful
        if analysis.get('insights'):
            update_note(f"Insight: {analysis['insights']}")
        
        # 4. Prompt Synthesis Phase
        # This is where the magic happens: merging short-term, long-term, and emotional context
        prompt = build_final_prompt(user_input, short_term_history, long_term_data, analysis)
        
        # 5. Generation Phase (AI)
        # Automatically uses Groq API if GROQ_API_KEY is in .env, otherwise Mock mode
        ai_response = get_ai_response(prompt)
        
        # 6. History Update Phase
        # We store both sides of the conversation for context in the NEXT turn
        add_to_history("user", user_input)
        add_to_history("assistant", ai_response)
        
        # 7. Output Phase
        print(f"AI: {ai_response}")

In [3]:
# Start the interaction!
if __name__ == "__main__":
    start_therapy_session()


--- [AI Therapist Session #4] ---
Hello, Harsh Pratap Singh. I am here to listen. (Type 'quit' to end)

AI: I'm glad you're here. How has your day been truly feeling?


You:  hey , i am fine just a little tired today can u help suggesting me some fun ways to relax ? 


AI: It sounds like you're feeling a bit drained and are looking for some ways to unwind, Harsh. I'm here to help you with that. Before we dive into some fun relaxation techniques, I want to acknowledge that it takes a lot of courage to admit when we're not feeling our best. It's completely normal to feel tired, and I'm glad you're taking the first step by reaching out for support.

I recall from our past conversations that you've been experiencing some sadness and anxiety related to work. I'm wondering, do you think your current fatigue might be connected to any work-related stress or pressures you've been facing lately? What are your thoughts on that, Harsh?


You:  bye


AI: You've made good progress today. Take care of yourself.


In [4]:
 import json

with open('memory.json','r') as f:
    data = json.load(f)

print("--- 🧠 AI LONG-TERM MEMORY ---")
print(f"User Name: {data.get('user_name')}")
print(f"Total Sessions: {data.get('session_count')}")
print("\nPsychological Insights Saved:")
for note in data.get('persistent_notes', []):
    print(f" -> {note}")

--- 🧠 AI LONG-TERM MEMORY ---
User Name: Harsh Pratap Singh
Total Sessions: 4

Psychological Insights Saved:
 -> Insight: {'emotions': ['sadness', 'anxiety'], 'patterns': [], 'concerns': ['work']}
